# Sampling-method emulator diagnostics

This notebook is a staging area for comparing emulator behavior under different parameter-sampling methods.

## Method 1: paper/range training region

The first diagnostic reproduces the parameter grid colored by relative `alpha_D` error from `test_strength_outside_train_region.ipynb`, using the current `Dipole_polarizability/runs_em1` emulator artifacts.

## Method 2: random train/test split

A placeholder cell is included at the end for the next sampling method.

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import tensorflow as tf
from matplotlib.colors import LogNorm

start = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [start, *start.parents]:
    if (candidate / "Dipole_polarizability").exists() and (candidate / "data/nuclear/160Yb_2d").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError("Could not find the SMLR repository root.")

DIPOLE_DIR = PROJECT_ROOT / "Dipole_polarizability"
sys.path.insert(0, str(DIPOLE_DIR / "src"))

import helper_gpt

plt.rcParams.update({"figure.dpi": 120})
print("Project root:", PROJECT_ROOT)
print("helper_gpt:", helper_gpt.__file__)


Project root: /Users/laurenjin/Documents/projects/SMLR/SMLR-jingy
helper_gpt: /Users/laurenjin/Documents/projects/SMLR/SMLR-jingy/Dipole_polarizability/src/helper_gpt.py


## Shared configuration

In [ ]:
RUN_DIR = DIPOLE_DIR / "runs_em1"
if not (RUN_DIR / "best_params_global.txt").exists():
    RUN_DIR = PROJECT_ROOT / "runs_em1"

STRENGTH_DIR = PROJECT_ROOT / "data/nuclear/160Yb_2d/total_strength"
ALPHAD_DIR = PROJECT_ROOT / "data/nuclear/160Yb_2d/total_alphaD"
STRENGTH_REGEX = r"strength_(?P<p2>[0-9.]+)_(?P<p1>[0-9.]+)\.out"
ALPHAD_REGEX = None

PARAMS_FILE = RUN_DIR / "best_params_global.txt"
SUMMARY_FILE = RUN_DIR / "run_summary.json"
summary = json.loads(SUMMARY_FILE.read_text()) if SUMMARY_FILE.exists() else {}

model = summary.get("model", {})
N = int(model.get("n", 13))
RETAIN = float(model.get("retain", 0.6))
ANSATZ = model.get("ansatz", "paper_dipole")
WIDTH_MODEL = model.get("width_model", "affine")
USE_VECTOR_TERMS = bool(model.get("use_vector_terms", True))
TRAIN_FILTER_RANGES = summary.get("train_filters") or {"p1": [0.4, 1.8], "p2": [1.5, 4.0]}
CENTRAL_POINT = summary.get("central_point")

print("Run dir:", RUN_DIR)
print("Params file:", PARAMS_FILE)
print("Model:", {"n": N, "retain": RETAIN, "ansatz": ANSATZ, "width_model": WIDTH_MODEL, "use_vector_terms": USE_VECTOR_TERMS})
print("Training filter ranges:", TRAIN_FILTER_RANGES)
print("Central point:", CENTRAL_POINT)


## Load full grid and current training-region points

In [ ]:
params = np.loadtxt(PARAMS_FILE).astype(np.float32)

dataset = helper_gpt.load_dataset(
    strength_dir=str(STRENGTH_DIR),
    alphaD_dir=str(ALPHAD_DIR),
    strength_regex=STRENGTH_REGEX,
    alphaD_regex=ALPHAD_REGEX,
    filter_ranges=None,
    central_point=CENTRAL_POINT,
)

train_param_file = RUN_DIR / "train_param_values.txt"
if train_param_file.exists():
    train_param_values = np.loadtxt(train_param_file, dtype=np.float32)
    if train_param_values.ndim == 1:
        train_param_values = train_param_values[None, :]
else:
    train_mask = np.ones(len(dataset.param_values), dtype=bool)
    for name, bounds in TRAIN_FILTER_RANGES.items():
        col = dataset.param_names.index(name)
        lo, hi = map(float, bounds)
        train_mask &= (dataset.param_values[:, col] >= lo) & (dataset.param_values[:, col] <= hi)
    train_param_values = dataset.param_values[train_mask]

print("Full grid samples:", len(dataset.strengths))
print("Parameter names:", dataset.param_names)
print("Dataset central point:", dataset.central_point.tolist())
print("Training-region samples:", train_param_values.shape[0])
print("Params shape:", params.shape)


## Evaluation helper

In [ ]:
def coerce_params_layout(params_np, config):
    params_np = np.asarray(params_np, dtype=np.float32).reshape(-1)
    layout = helper_gpt.get_packed_layout(config)
    if params_np.size == layout.total_size:
        return params_np
    raise ValueError(f"Expected parameter vector of length {layout.total_size}, got {params_np.size}.")


def evaluate_alphaD_on_grid(dataset, params_np, n, retain, ansatz, width_model, use_vector_terms=True):
    config = helper_gpt.AnsatzConfig(
        n=n,
        n_params=int(dataset.param_values.shape[1]),
        ansatz=ansatz,
        width_model=width_model,
        use_vector_terms=use_vector_terms,
    )
    params_np = coerce_params_layout(params_np, config)

    M_batch, v_batch, _, _ = helper_gpt.build_model_matrices_and_vectors(
        params=tf.convert_to_tensor(params_np, dtype=tf.float32),
        config=config,
        param_values=tf.convert_to_tensor(dataset.param_values, dtype=tf.float32),
        central_point=tf.convert_to_tensor(dataset.central_point, dtype=tf.float32),
    )
    eigenvalues, eigenvectors = tf.linalg.eigh(M_batch)

    n_i = int(eigenvalues.shape[1])
    k_keep = max(1, min(int(round(retain * n_i)), n_i))
    left = (n_i - k_keep) // 2
    right = left + k_keep

    eigvals_kept = eigenvalues[:, left:right]
    eigvecs_kept = eigenvectors[:, :, left:right]
    proj = tf.matmul(tf.transpose(eigvecs_kept, perm=[0, 2, 1]), v_batch[:, :, None])
    B_batch = tf.square(tf.squeeze(proj, axis=-1))

    alphaD_pred = []
    for i in range(len(dataset.strengths)):
        alphaD_pred.append(helper_gpt.calculate_alphaD(eigvals_kept[i].numpy(), B_batch[i].numpy()))
    return np.asarray(alphaD_pred, dtype=float)


## Method 1: parameter grid colored by `alpha_D` relative error

In [ ]:
alphaD_true = np.asarray(dataset.alphaD_values, dtype=float)
alphaD_pred = evaluate_alphaD_on_grid(
    dataset=dataset,
    params_np=params,
    n=N,
    retain=RETAIN,
    ansatz=ANSATZ,
    width_model=WIDTH_MODEL,
    use_vector_terms=USE_VECTOR_TERMS,
)
alphaD_rel_err = np.abs(alphaD_pred - alphaD_true) / np.maximum(np.abs(alphaD_true), 1e-12)

print("Mean alphaD relative error:", float(np.mean(alphaD_rel_err)))
print("Median alphaD relative error:", float(np.median(alphaD_rel_err)))
print("Max alphaD relative error:", float(np.max(alphaD_rel_err)))

x = dataset.param_values[:, 0]
y = dataset.param_values[:, 1]
color = np.clip(alphaD_rel_err, 1e-12, None)

fig, ax = plt.subplots(figsize=(7, 5.5))
sc = ax.scatter(x, y, c=color, marker="s", cmap="Spectral", norm=LogNorm(), s=55)
fig.colorbar(sc, ax=ax, label=r"relative error $\alpha_D$")

if len(x) <= 250:
    for i, (xi, yi) in enumerate(zip(x, y)):
        ax.text(xi, yi, str(i), ha="center", va="center", fontsize=5.5, color="black")

train_x = train_param_values[:, 0]
train_y = train_param_values[:, 1]
rect = patches.Rectangle(
    (float(np.min(train_x)), float(np.min(train_y))),
    float(np.max(train_x) - np.min(train_x)),
    float(np.max(train_y) - np.min(train_y)),
    linewidth=1.5,
    edgecolor="black",
    facecolor="none",
)
ax.add_patch(rect)
ax.set_xlabel(dataset.param_names[0])
ax.set_ylabel(dataset.param_names[1])
ax.set_title(r"Method 1: range-trained emulator, $\alpha_D$ relative error")
plt.show()


## Method 2: random train/test split

Placeholder for the next sampling method.